In [4]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# ============================================================
# CONFIG
# ============================================================

EXCEL_PATH = Path("/content/daq_raw_4blends.xlsx")  # change if needed
OUTPUT_GRAPH = Path("Pressure_vs_Time.png")

SHEETS_TO_USE = None          # None = first 4 sheets
REQUIRED_SHEET_COUNT = 4

TIME_COL = "X_Value"          # time (s)
PRESSURE_COL = "Pressure"     # change to "Pressure_0" if that is your channel

# ============================================================
# LOAD + PLOT
# ============================================================

def load_sheets(excel_path: Path):
    if not excel_path.exists():
        raise FileNotFoundError(f"Excel not found: {excel_path.resolve()}")

    xls = pd.ExcelFile(excel_path, engine="openpyxl")
    sheet_names = xls.sheet_names if SHEETS_TO_USE is None else list(SHEETS_TO_USE)
    sheet_names = sheet_names[:REQUIRED_SHEET_COUNT]

    frames = {}
    for sheet in sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet, engine="openpyxl")

        missing = [c for c in (TIME_COL, PRESSURE_COL) if c not in df.columns]
        if missing:
            raise ValueError(
                f"Sheet '{sheet}' missing columns {missing}. "
                f"Present: {list(df.columns)}"
            )

        sub = df[[TIME_COL, PRESSURE_COL]].copy()
        sub[TIME_COL] = pd.to_numeric(sub[TIME_COL], errors="coerce")
        sub[PRESSURE_COL] = pd.to_numeric(sub[PRESSURE_COL], errors="coerce")
        sub = sub.dropna()
        sub = sub[np.isfinite(sub[TIME_COL]) & np.isfinite(sub[PRESSURE_COL])]
        sub = sub[sub[TIME_COL] >= 0].sort_values(TIME_COL)

        frames[sheet] = sub
        print(f"{sheet}: {len(sub)} points")

    return frames, sheet_names


def plot_pressure_vs_time(frames, sheet_names, output_path: Path):
    fig, ax = plt.subplots(figsize=(10, 6))

    for sheet in sheet_names:
        df = frames[sheet]
        if df.empty:
            print(f"Skipping empty sheet: {sheet}")
            continue
        ax.plot(df[TIME_COL], df[PRESSURE_COL], linewidth=1.5, alpha=0.85, label=str(sheet))

    ax.set_xlabel("Time")
    ax.set_ylabel("Pressure")
    ax.set_title("Pressure vs Time (all firings)")
    ax.grid(True, alpha=0.3)
    ax.legend(title="Sheet", fontsize=8)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)
    print(f"Graph saved: {output_path.resolve()}")


def main():
    frames, sheet_names = load_sheets(EXCEL_PATH)
    plot_pressure_vs_time(frames, sheet_names, OUTPUT_GRAPH)


if __name__ == "__main__":
    main()

Firing 01: 12000 points
Firing 02: 11000 points
Firing 03: 14000 points
Firing 04: 14000 points
Graph saved: /content/Pressure_vs_Time.png
